# 🦌 Doñana — 2. Entrenamiento de un detector de animales con PyTorch

Adaptación del notebook de entrenamiento de clase (robots) al dataset de Doñana. Cambios principales:

| Plantilla (robots) | Doñana |
|---|---|
| 1 clase (`num_classes = 2`) | Una clase por especie/familia + fondo |
| Anotaciones XML por secuencia | `processed/annotations_*.csv` (de `01_preparar_datos.ipynb`) |
| `random_split` 80/20 | Split fijo por ráfagas y estratificado por clase (el test no se toca hasta `03_test`) |
| `Normalize(mean, std)` en el dataset | Solo `[0, 1]`: el modelo de torchvision ya normaliza internamente (normalizar dos veces perjudica) |
| `torch.stack(images)` | Lista de imágenes (pueden tener tamaños distintos) |
| Guarda el modelo con menor pérdida de **train** | Guarda el de mayor **mAP en validación** + `last.pt` para reanudar |

El código común (dataset, modelos, bucle de entrenamiento, métricas) está en `donana_det.py`, para reutilizarlo en
`04_data_augmentation.ipynb` exactamente igual.

## Instalación e imports
*Entorno de ejecución → Cambiar tipo → GPU (T4)*.

In [ ]:
!pip install -q -U "albumentations>=2.0.5" torchmetrics pycocotools

In [ ]:
import sys
from pathlib import Path

RUNNING_IN_COLAB = "google.colab" in sys.modules
if RUNNING_IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path("/content/drive/MyDrive/donana")  # donana_data.py, donana_det.py y processed/ están aquí
    LOCAL_DIR = Path("/content")                         # disco local de Colab (rápido)
else:
    PROJECT_DIR = Path.cwd()
    LOCAL_DIR = Path.cwd()
sys.path.append(str(PROJECT_DIR))

PROCESSED_DIR = PROJECT_DIR / "processed"            # anotaciones, splits y caché comprimida
MODELS_DIR = PROJECT_DIR / "models"                  # checkpoints (en Drive, sobreviven a desconexiones)
IMAGES_DIR = LOCAL_DIR / "dona_24_1024"              # imágenes reducidas (lado mayor 1024 px)
CACHE_ZIP = PROCESSED_DIR / "dona_24_1024.zip"

import json
import numpy as np
import pandas as pd
import torch
import donana_data as dd
import donana_det as det

dd.ensure_image_cache(IMAGES_DIR, CACHE_ZIP)
annotations = pd.read_csv(PROCESSED_DIR / "annotations.csv")
CLASS_NAMES = json.loads((PROCESSED_DIR / "dataset_info.json").read_text())["class_names"]
NUM_CLASSES = len(CLASS_NAMES) + 1   # + fondo (clase 0)
train_df = annotations[annotations["split"] == "train"]
valid_df = annotations[annotations["split"] == "valid"]
test_df = annotations[annotations["split"] == "test"]
print(f"Clases ({NUM_CLASSES - 1}): {CLASS_NAMES}")
print(f"train {len(train_df)} | valid {len(valid_df)} | test {len(test_df)}")

## Parámetros

| Parámetro | Qué hace |
|---|---|
| `model_name` | Arquitectura. `faster_rcnn_v1` (ResNet50-FPN) es una buena base; `faster_rcnn_v3` (MobileNetV3) es ~3-4× más rápido pero menos preciso. |
| `image_size` | Lado mayor con el que el modelo procesa la imagen. Más grande ayuda con animales pequeños/lejanos (conejos, ginetas), pero es más lento. |
| `num_epochs` | Pasadas completas por el train. Con `early_stopping_patience` se para antes si el mAP de validación deja de mejorar. |
| `batch_size` | Imágenes por paso. Limitado por la memoria de la GPU (4 cabe bien en una T4 a 1024 px con AMP). |
| `lr` | Learning rate del SGD. Regla de torchvision: 0.02 para batch 16 → 0.005 para batch 4. |
| `lr_momentum` | Inercia del SGD: promedia los gradientes recientes y suaviza la actualización. |
| `lr_decay` | *Weight decay* (regularización L2): penaliza pesos grandes para reducir sobreajuste. |
| `lr_factor`, `lr_patience`, `lr_threshold`, `lr_min` | `ReduceLROnPlateau`: si el mAP de validación no mejora más de `lr_threshold` en `lr_patience` épocas, `lr *= lr_factor` (hasta `lr_min`). |
| `max_grad_norm` | Recorte de la norma del gradiente (0 = sin recorte). Útil en RetinaNet/SSD, que a veces dan pérdidas `nan` al principio. |
| `use_amp` | Precisión mixta en GPU: ~2× más rápido y menos memoria. |

In [ ]:
seed = 0
det.set_seed(seed)

# Modelo
model_name = "faster_rcnn_v1"
pretrained = True                    # pesos preentrenados en COCO
image_size = 1024

# Entrenamiento
use_gpu = True
use_amp = True
num_workers = 2                      # Colab tiene 2 CPU
num_epochs = 15
early_stopping_patience = 5
batch_size = 4
lr = 0.005
lr_momentum = 0.9
lr_decay = 1e-4
lr_factor = 0.1
lr_patience = 2
lr_threshold = 1e-3
lr_min = 1e-5
models_to_clip = ["retinanet_v1", "retinanet_v2", "ssd_v1", "ssd_v2"]
max_grad_norm = 1.0 if model_name in models_to_clip else 0.0

experiment_name = f"{model_name}_baseline"
output_dir = MODELS_DIR / experiment_name
config = dict(model_name=model_name, num_classes=NUM_CLASSES, image_size=image_size, class_names=CLASS_NAMES,
              batch_size=batch_size, lr=lr, augmentation=None, num_augments=0, seed=seed)

## Datos
Dataset = cómo cargar **una** imagen y su anotación; DataLoader = cómo agruparlas en batches (y en paralelo).

In [ ]:
train_set = det.DonanaDataset(train_df, IMAGES_DIR)      # sin aumentación (baseline)
valid_set = det.DonanaDataset(valid_df, IMAGES_DIR)

train_loader = det.make_loader(train_set, batch_size, shuffle=True, num_workers=num_workers, seed=seed)
valid_loader = det.make_loader(valid_set, batch_size, shuffle=False, num_workers=num_workers)

image, target = train_set[0]
print(image.shape, target)

## Modelo, optimizador y scheduler

In [ ]:
device = det.get_device(use_gpu)
model = det.torchvision_model(model_name, NUM_CLASSES, pretrained=pretrained, image_size=image_size).to(device)
optimizer = det.build_optimizer(model, lr=lr, momentum=lr_momentum, weight_decay=lr_decay)
lr_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="max", factor=lr_factor, patience=lr_patience, threshold=lr_threshold, min_lr=lr_min
)
print(f"Dispositivo: {device} | salida: {output_dir}")

## Entrenamiento
En cada época: entrenamiento → pérdida de validación → mAP de validación. Si Colab se desconecta, vuelve a ejecutar
el notebook: `fit` detecta `last.pt` y **reanuda** donde lo dejó.

El bucle (en `det.train_one_epoch`) es el mismo de la plantilla con los huecos rellenados:
```python
loss_dict = model(images, targets)
loss = sum(loss_dict.values())
optimizer.zero_grad()        # Reset gradients
loss.backward()              # Backpropagation
clip_grad_norm_(...)         # Clip gradients
optimizer.step()             # Update weights
```

In [ ]:
history = det.fit(
    model, optimizer, lr_scheduler, train_loader, valid_loader, device,
    num_epochs=num_epochs, output_dir=output_dir, config=config,
    early_stopping_patience=early_stopping_patience, use_amp=use_amp, max_grad_norm=max_grad_norm,
)

In [ ]:
det.plot_history(history, save_path=output_dir / "history.png")

**Cómo leer la gráfica:** si la pérdida de train sigue bajando pero la de validación sube o el mAP de validación se
estanca, el modelo está **sobreajustando** (memoriza el train). Ese es justo el caso en el que la aumentación de datos
(`04_data_augmentation.ipynb`) suele ayudar.

## Resultado en validación (mejor modelo)

In [ ]:
best_model, _ = det.load_trained_model(output_dir / "best.pt", device)
valid_results = det.evaluate(best_model, valid_loader, device, CLASS_NAMES)
print(f"Mejor época: {history['best_epoch']}")
pd.Series(det.summary_row(valid_results)).round(4).to_frame("valid")